# Marine Acoustic Monitoring — Data Explorer

This notebook provides tools for loading, exploring, and visualizing underwater acoustic data from SoundTrap hydrophones deployed in the Bay of San Cristóbal, Galápagos.

**What's here:**
0. Data download from R2 cloud storage (works on Colab, Kaggle, RunPod)
1. Dataset inventory — discover what recordings are available
2. Audio loading — read WAV files with filtering
3. Spectrogram visualization — full-band and frequency-band views
4. XML metadata parsing — deployment info from SoundTrap logs
5. In-notebook listening — play audio clips directly

All functions are also available as a standalone module: `import acoustic_data as ad`

## 0. Setup & Data Download

Run these cells to install dependencies, configure R2 credentials, and download
the marine acoustic dataset. Works on Colab, Kaggle, RunPod, and local.

**Credentials:** Organizers will provide a `participant-download.env` file. Upload it
to the Colab file panel (drag and drop), then run the next cells. Alternatively, edit
the placeholder values directly.

In [ ]:
!pip install -q boto3 tqdm soundfile scipy

In [ ]:
import os
from pathlib import Path

# === Get r2_download.py helper module (if running on Colab/Kaggle) ===
HELPER_URL = "https://raw.githubusercontent.com/SALA-AI-LATAM/hackathon-participants/main/r2_download.py"

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

# === Set R2 credentials ===
# Search common locations for the .env file
ENV_NAME = "participant-download.env"
ENV_SEARCH_PATHS = [
    Path(ENV_NAME),               # current directory (notebook folder)
    Path("..") / ENV_NAME,        # parent directory (repo root)
    Path("/workspace") / ENV_NAME,  # RunPod workspace root
]

env_file = None
for p in ENV_SEARCH_PATHS:
    if p.exists():
        env_file = p
        break

if env_file is not None:
    for line in open(env_file):
        line = line.strip()
        if line and not line.startswith("#"):
            key, val = line.removeprefix("export ").split("=", 1)
            os.environ[key] = val.strip('"')
    print(f"Credentials loaded from {env_file}")
elif "R2_ENDPOINT" in os.environ:
    print("Using pre-set environment variables")
else:
    # Option B: Paste credentials directly (organizers will provide these)
    os.environ["R2_ENDPOINT"] = "https://YOUR_ACCOUNT_ID.r2.cloudflarestorage.com"
    os.environ["R2_ACCESS_KEY_ID"] = "YOUR_ACCESS_KEY"
    os.environ["R2_SECRET_ACCESS_KEY"] = "YOUR_SECRET_KEY"
    os.environ["R2_BUCKET"] = "sala-2026-hackathon-data"
    print("Using inline credentials (edit the values above if they say YOUR_...)")

In [ ]:
import r2_download as hd
import tarfile

print(f"Environment: {hd._detect_environment()}")
data_dir = hd._default_data_dir()
print(f"Data directory: {data_dir}")

# === Download marine acoustic data from R2 ===
client = hd.get_s3_client()

# Force fresh manifest (don't use stale cache)
if Path("manifest.json").exists():
    Path("manifest.json").unlink()

manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"], s3_client=client, cache_path="manifest.json"
)
hd.summarize_manifest(manifest)

# Colab-friendly subset: single 425 MB tar.gz with 11 WAVs covering all 3 units
# Switch to "marine-acoustic-core" for the full 7.3 GB dataset (local/RunPod)
DATASET = "marine-acoustic-colab"

stats = hd.download_dataset(manifest, dataset_name=DATASET)
print(f"\nDownloaded: {stats['downloaded']}, Skipped: {stats['skipped']}, Failed: {stats['failed']}")

# Extract tar.gz if we downloaded the colab subset
if DATASET == "marine-acoustic-colab":
    tar_path = Path(data_dir) / "marine-acoustic-colab" / "marine-acoustic-colab.tar.gz"
    extract_dir = Path(data_dir)
    marker = extract_dir / "marine-acoustic" / ".extracted"
    if tar_path.exists() and not marker.exists():
        print(f"\nExtracting {tar_path.name}...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=extract_dir)
        marker.touch()
        print(f"Extracted to {extract_dir / 'marine-acoustic'}/")
    elif marker.exists():
        print("\nAlready extracted.")
    else:
        print(f"\nWarning: tar.gz not found at {tar_path}")

In [ ]:
import os
import re
import wave
import glob
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, spectrogram as sp_spectrogram

try:
    import soundfile as sf
except ImportError:
    sf = None
    print("soundfile not available — using stdlib wave module (works fine)")

%matplotlib inline

In [ ]:
# === Data directory (set automatically from R2 download location) ===
# After R2 download, files land at {data_dir}/marine-acoustic/{unit}/...
import r2_download as hd
DATA_DIR = str(Path(hd._default_data_dir()) / "marine-acoustic")
print(f"Data directory: {DATA_DIR}")
print(f"Expected subdirs: 5783/, 6478/, Music_Soundtrap_Pilot/")

## 1. Dataset Inventory

Three SoundTrap units with different sample rates:

| Unit | Sample Rate | WAV Files | Duration/File | Frequency Range |
|------|------------|-----------|---------------|-----------------|
| 5783 | 144 kHz | 16 | 20 min | Up to 72 kHz (echolocation) |
| 6478 | 96 kHz | 189 | 10 min | Up to 48 kHz |
| Pilot | 48 kHz | 721 | ~5 min | Up to 24 kHz |

In [ ]:
# === Dataset constants ===

UNITS = {
    "5783": {
        "sample_rate": 144_000,
        "description": "SoundTrap unit 5783, 144 kHz, 20-min recordings",
    },
    "6478": {
        "sample_rate": 96_000,
        "description": "SoundTrap unit 6478, 96 kHz, 10-min recordings",
    },
    "pilot": {
        "sample_rate": 48_000,
        "description": "Pilot deployment, 48 kHz, ~5-min recordings",
    },
}

# Map directory names to unit keys
_DIR_TO_UNIT = {
    "5783": "5783",
    "6478": "6478",
    "Music_Soundtrap_Pilot": "pilot",
}

In [ ]:
# === Timestamp parsing ===
# SoundTrap filenames encode the recording start time

def parse_soundtrap_timestamp(filename):
    """Extract UTC datetime from a SoundTrap WAV filename."""
    name = Path(filename).name

    # Unit 5783 / 6478: {unit_id}.{YYMMDDHHMMSS}.{ext}
    m = re.match(r"^\d{4}\.(\d{12})\.wav$", name)
    if m:
        try:
            return datetime.strptime(m.group(1), "%y%m%d%H%M%S")
        except ValueError:
            return None

    # Pilot: {YYMMDD}_{sequence}.wav — date only, no time
    m = re.match(r"^(\d{6})_(\d+)\.wav$", name)
    if m:
        try:
            return datetime.strptime(m.group(1), "%y%m%d")
        except ValueError:
            return None

    return None

In [ ]:
# === Dataset discovery ===

def list_recordings(data_dir, unit=None):
    """List all WAV recordings with parsed metadata.
    
    Returns list of dicts: path, filename, unit, timestamp, sample_rate
    """
    data_dir = Path(data_dir)
    recordings = []

    dirs = _DIR_TO_UNIT.items()
    if unit:
        dirs = [(k, v) for k, v in dirs if v == unit]

    for dirname, unit_key in dirs:
        unit_dir = data_dir / dirname
        if not unit_dir.exists():
            continue

        unit_info = UNITS[unit_key]
        # Filter out macOS resource fork files (._*)
        wavs = sorted(p for p in unit_dir.glob("*.wav") if not p.name.startswith("._"))

        for wav_path in wavs:
            ts = parse_soundtrap_timestamp(wav_path.name)
            recordings.append({
                "path": wav_path,
                "filename": wav_path.name,
                "unit": unit_key,
                "timestamp": ts,
                "sample_rate": unit_info["sample_rate"],
            })

    return recordings


def inventory(data_dir):
    """Print a summary of available recordings per unit."""
    data_dir = Path(data_dir)
    recs = list_recordings(data_dir)

    print(f"{'Unit':<8} {'WAVs':>6} {'Sample Rate':>12} {'Time Range'}")
    print("-" * 60)

    for unit_key in ["5783", "6478", "pilot"]:
        unit_recs = [r for r in recs if r["unit"] == unit_key]
        if not unit_recs:
            continue
        sr = unit_recs[0]["sample_rate"]
        timestamps = [r["timestamp"] for r in unit_recs if r["timestamp"]]
        if timestamps:
            t_min = min(timestamps).strftime("%Y-%m-%d")
            t_max = max(timestamps).strftime("%Y-%m-%d")
            time_range = f"{t_min} → {t_max}"
        else:
            time_range = "unknown"
        print(f"{unit_key:<8} {len(unit_recs):>6} {sr:>10} Hz  {time_range}")

    print(f"\nTotal: {len(recs)} WAV files")
    return recs

In [ ]:
# === Run inventory ===
recs = inventory(DATA_DIR)

## 2. Audio Loading & Filtering

In [ ]:
def load_audio(path, duration_s=None, offset_s=0.0, target_sr=None):
    """Load a WAV file (or a segment) as a float32 numpy array.
    
    Returns (audio, sample_rate) — audio is float32 in [-1, 1].
    """
    path = str(path)

    if sf is not None:
        info = sf.info(path)
        sr = info.samplerate
        start_frame = int(offset_s * sr)
        n_frames = int(duration_s * sr) if duration_s is not None else -1
        audio, sr = sf.read(path, start=start_frame, frames=n_frames, dtype="float32")
    else:
        # Fallback: stdlib wave module
        with wave.open(path, "rb") as wf:
            sr = wf.getframerate()
            n_channels = wf.getnchannels()
            n_total = wf.getnframes()
            start_frame = int(offset_s * sr)
            if duration_s is not None:
                n_frames = min(int(duration_s * sr), n_total - start_frame)
            else:
                n_frames = n_total - start_frame
            wf.setpos(start_frame)
            raw = wf.readframes(n_frames)
        audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
        if n_channels > 1:
            audio = audio.reshape(-1, n_channels).mean(axis=1)

    # Optional resampling (requires librosa)
    if target_sr and target_sr != sr:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    return audio, sr


def highpass_filter(audio, sr, cutoff_hz=50, order=4):
    """Remove DC offset and low-frequency self-noise (default cutoff: 50 Hz)."""
    sos = butter(order, cutoff_hz, btype="highpass", fs=sr, output="sos")
    return sosfilt(sos, audio).astype(np.float32)

In [ ]:
# === Load and filter a 30-second clip from unit 6478 ===
recs_6478 = [r for r in recs if r["unit"] == "6478"]
r = recs_6478[len(recs_6478) // 2]  # pick one from the middle

audio, sr = load_audio(r["path"], duration_s=30.0)
audio = highpass_filter(audio, sr, cutoff_hz=50)

print(f"File: {r['filename']}")
print(f"Sample rate: {sr} Hz | Duration: {len(audio)/sr:.1f}s | Shape: {audio.shape}")
print(f"Amplitude range: [{audio.min():.3f}, {audio.max():.3f}]")

## 3. Spectrogram Visualization

Spectrograms show frequency content over time — the primary visualization tool for underwater acoustics.

In [ ]:
def compute_spectrogram(audio, sr, n_fft=2048, hop_length=512, f_min=0, f_max=None):
    """Compute power spectrogram in dB.
    
    Returns (S_db, freqs, times).
    """
    freqs, times, Sxx = sp_spectrogram(
        audio, fs=sr, nperseg=n_fft, noverlap=n_fft - hop_length,
        scaling="spectrum",
    )

    # Convert to dB
    Sxx_db = 10 * np.log10(Sxx + 1e-12)

    # Frequency band selection
    f_max = f_max or sr / 2
    freq_mask = (freqs >= f_min) & (freqs <= f_max)

    return Sxx_db[freq_mask, :], freqs[freq_mask], times


def plot_spectrogram(audio, sr, title=None, duration_s=None, f_max=None,
                     n_fft=2048, hop_length=512, figsize=(14, 4),
                     cmap="magma", vmin=None, vmax=None, ax=None):
    """Plot a spectrogram from raw audio."""
    if duration_s is not None:
        audio = audio[: int(duration_s * sr)]

    S_db, freqs, times = compute_spectrogram(
        audio, sr, n_fft=n_fft, hop_length=hop_length, f_max=f_max
    )

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(1, 1, figsize=figsize)

    im = ax.pcolormesh(times, freqs / 1000, S_db,
                       shading="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_ylabel("Frequency (kHz)")
    ax.set_xlabel("Time (s)")
    if title:
        ax.set_title(title)
    plt.colorbar(im, ax=ax, label="Power (dB)")

    if created_fig:
        fig.tight_layout()
        return fig, ax
    return ax


def plot_spectrogram_bands(audio, sr, title_prefix="", figsize=(14, 10)):
    """Plot spectrograms at ecologically relevant frequency bands.
    
    - LOW (50-2000 Hz): Fish vocalizations, boat noise
    - MID (2-20 kHz): Snapping shrimp, dolphin whistles
    - HIGH (20+ kHz): Echolocation clicks (only if sample rate supports it)
    """
    nyquist = sr / 2
    bands = [
        ("LOW (50-2000 Hz): fish, boats", 50, 2000, 1024),
        ("MID (2-20 kHz): shrimp, dolphin whistles", 2000, 20000, 2048),
    ]
    if nyquist > 24000:
        bands.append(
            (f"HIGH (20-{nyquist/1000:.0f} kHz): echolocation", 20000, nyquist, 4096)
        )

    fig, axes = plt.subplots(len(bands), 1, figsize=figsize)
    if len(bands) == 1:
        axes = [axes]

    for ax, (label, f_min, f_max, nfft) in zip(axes, bands):
        f_max = min(f_max, nyquist)
        plot_spectrogram(
            audio, sr, title=f"{title_prefix}{label}",
            f_max=f_max, n_fft=nfft, ax=ax,
        )
        ax.set_ylim(f_min / 1000, f_max / 1000)

    fig.tight_layout()
    return fig, axes

### 3.1 Full-band spectrogram

In [ ]:
plot_spectrogram(audio, sr, title=f"Unit 6478 — {r['filename']} ({sr/1000:.0f} kHz sample rate)");

### 3.2 Frequency-band split

Ecological frequency bands for marine soundscapes:
- **LOW (50–2000 Hz)**: Fish choruses, boat engine noise
- **MID (2–20 kHz)**: Snapping shrimp, dolphin whistles
- **HIGH (20+ kHz)**: Echolocation clicks (only visible at high sample rates)

In [ ]:
plot_spectrogram_bands(audio, sr, title_prefix="6478: ");

### 3.3 Compare across units

Load a clip from each unit to see how sample rate affects what's visible.

In [ ]:
# === Compare 10-second clips across all three units ===
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, unit_key in zip(axes, ["5783", "6478", "pilot"]):
    unit_recs = [r for r in recs if r["unit"] == unit_key]
    if not unit_recs:
        ax.set_title(f"{unit_key}: no recordings found")
        continue
    rec = unit_recs[len(unit_recs) // 2]
    clip, clip_sr = load_audio(rec["path"], duration_s=10.0)
    clip = highpass_filter(clip, clip_sr)
    plot_spectrogram(clip, clip_sr,
                     title=f"{unit_key} — {rec['filename']} ({clip_sr/1000:.0f} kHz)",
                     ax=ax)

fig.tight_layout();

## 4. XML Metadata

Each recording has a companion `.log.xml` file with deployment metadata: timestamps, sample rate, water temperature, battery voltage, gain settings.

In [ ]:
def parse_xml_metadata(xml_path):
    """Parse a SoundTrap .log.xml file for deployment metadata.
    
    Returns dict with available fields: start_time, stop_time, sample_rate,
    temperature_c, battery_v, gain_db, hardware_id.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    meta = {}

    for pe in root.findall(".//PROC_EVENT"):
        for child in pe:
            attrs = child.attrib
            if "SamplingStartTimeUTC" in attrs:
                try:
                    meta["start_time"] = datetime.strptime(
                        attrs["SamplingStartTimeUTC"], "%Y-%m-%dT%H:%M:%S"
                    )
                except ValueError:
                    pass
            if "SamplingStopTimeUTC" in attrs:
                try:
                    meta["stop_time"] = datetime.strptime(
                        attrs["SamplingStopTimeUTC"], "%Y-%m-%dT%H:%M:%S"
                    )
                except ValueError:
                    pass
            if "SampleRate" in attrs:
                meta["sample_rate"] = int(attrs["SampleRate"])
            if "Temperature" in attrs:
                meta["temperature_c"] = float(attrs["Temperature"])
            if "BatteryState" in attrs:
                meta["battery_v"] = float(attrs["BatteryState"])
            if "Gain" in attrs:
                meta["gain_db"] = float(attrs["Gain"])

    # Hardware ID from top-level
    hw = root.find(".//HARDWARE")
    if hw is not None and "SerialNumber" in hw.attrib:
        meta["hardware_id"] = hw.attrib["SerialNumber"]

    return meta

In [ ]:
# === Parse a few XML files from unit 6478 ===
xml_dir = Path(DATA_DIR) / "6478"
xml_files = sorted(xml_dir.glob("*.log.xml"))[:5]

for xf in xml_files:
    meta = parse_xml_metadata(xf)
    print(f"{xf.name}:")
    for k, v in meta.items():
        print(f"  {k}: {v}")
    print()

## 5. Listening

Play audio clips directly in the notebook. Audio is automatically resampled for browser playback.

**What you'll likely hear:** These recordings contain a mix of ambient ocean noise, boat engines, sensor self-noise, and equipment calibration tones. You may *not* hear obvious animal sounds — marine biologists have not yet confirmed cetacean presence in this particular dataset. That's okay: **this notebook is a walkthrough of tools** for analyzing underwater audio. The same pipeline works whether the recordings contain animal vocalizations or not — and the tools you build here are valuable for future deployments where animals are confirmed to be present.

In [ ]:
from IPython.display import Audio, display

def listen(path, duration_s=10, offset_s=0):
    """Play an audio clip in the notebook (resamples to 22050 Hz for browser)."""
    audio, sr = load_audio(path, duration_s=duration_s, offset_s=offset_s)

    # Resample for browser playback
    if sr > 22050:
        try:
            import librosa
            audio = librosa.resample(audio, orig_sr=sr, target_sr=22050)
            sr = 22050
        except ImportError:
            # Crude decimation fallback
            factor = sr // 22050
            audio = audio[::factor]
            sr = sr // factor

    display(Audio(audio, rate=sr))

In [ ]:
# === Listen to 10 seconds from each unit ===
for unit_key in ["6478", "pilot", "5783"]:
    unit_recs = [r for r in recs if r["unit"] == unit_key]
    if unit_recs:
        rec = unit_recs[len(unit_recs) // 2]
        print(f"Unit {unit_key}: {rec['filename']}")
        listen(rec["path"], duration_s=10)

## 6. Your Turn

Some things to try:

- **Browse recordings**: Change the index in `unit_recs[...]` to explore different files. Listen and look at spectrograms — can you spot boat noise, clicks, whistles?
- **Zoom in**: Use `offset_s` and `duration_s` in `load_audio()` to zoom into interesting time windows
- **Change frequency focus**: Pass `f_max=5000` to `plot_spectrogram()` to zoom into low frequencies
- **Batch analysis**: Loop over all recordings in a unit and compute statistics (mean power per band, peak frequency, etc.)

See **README.md** for full project ideas across three tiers:
1. **Acoustic indices** — characterize the soundscape with no labels
2. **Pretrained models** — zero-shot and few-shot classification with Perch/BirdNET
3. **Active learning** — bootstrap a classifier from scratch with minimal labels
4. **Bonus: Project CETI** — generative models for marine vocalizations (WhAM/VampNet)